In [5]:
import numpy as np

class Detector:
    def __init__(self, R, r, N):
        self.R = R
        self.r = r
        self.N = N
        self.positions = self._place_pmt()

    def _place_pmt(self):
        points = []
        while len(points) < self.N:
            p = np.random.uniform(-1, 1, size=3)
            length = np.linalg.norm(p)
            if 0 < length <= 1:
                points.append(p / length * self.R)
        return np.array(points)

    def check(self, point):
        point = np.array(point)
        distance_to_center = np.linalg.norm(point)
        difference = abs(distance_to_center - self.R)
        
        if difference > 0.05:
            return "Точка не на поверхности сферы!"

        for i, center in enumerate(self.positions):
            dist = np.linalg.norm(point - center)
            if dist <= self.r: 
                return f"Точка попала в ФЭУ номер {i}!"
                
        return "Точка на сфере, но не ФЭУ."
det = Detector(R=2.0, r=0.1, N=40)
test_point = det.positions[0]
result = det.check(test_point)

print(f"Проверяемая точка (центр ФЭУ 0): {test_point}")
print(f"Результат проверки: {result}")

Проверяемая точка (центр ФЭУ 0): [ 1.04610784e-03 -1.97498065e+00  3.15357461e-01]
Результат проверки: Точка попала в ФЭУ номер 0!


In [6]:
import k3d
import numpy as np


plot = k3d.plot()

plot += k3d.points(det.positions, 
                   color=0xff0000, 
                   point_size=0.1, 
                   shader='spherical', 
                   name='ФЭУ')

u = np.linspace(0, 2 * np.pi, 50)
v = np.linspace(0, np.pi, 50)
x = det.R * np.outer(np.cos(u), np.sin(v))
y = det.R * np.outer(np.sin(u), np.sin(v))
z = det.R * np.outer(np.ones(np.size(u)), np.cos(v))

vertices = np.column_stack([x.flatten(), y.flatten(), z.flatten()])
indices = []
for i in range(len(u) - 1):
    for j in range(len(v) - 1):
        p1 = i * len(v) + j
        p2 = p1 + 1
        p3 = (i + 1) * len(v) + j
        p4 = p3 + 1
        indices.append([p1, p2, p4])
        indices.append([p1, p4, p3])

plot += k3d.mesh(vertices.astype(np.float32), 
                 np.array(indices, dtype=np.uint32), 
                 color=0x888888, 
                 opacity=0.2, 
                 wireframe=False)

plot.display()

Output()